In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.utils import resample
from collections import defaultdict

In [2]:
# TCVファイルの読み込み
train_data = pd.read_csv('../000.data/train/train_D.tsv', sep="\t")
test = pd.read_csv('../000.data/test.tsv', sep="\t")

In [3]:
# 末尾が "_D" のテストデータだけを抽出
test_data = test[test["user_id"].str.endswith("_D")]

In [4]:
# 関連度スコアの設定
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:
        return 3  # 広告経由の購入
    elif row["event_type"] == 2:
        return 2  # 広告クリック
    elif row["event_type"] == 1:
        return 1  # 詳細ページ閲覧
    else:
        return 0  # カート追加

train_data["relevance"] = train_data.apply(compute_relevance, axis=1)

In [5]:
# タイムスタンプ処理
train_data["time_stamp"] = pd.to_datetime(train_data["time_stamp"])
train_data["time_since_last"] = (train_data["time_stamp"].max() - train_data["time_stamp"]).dt.total_seconds()

In [6]:
# ユーザーと商品の特徴量
user_features = train_data.groupby("user_id").agg(
    user_total_interactions=("product_id", "count"),
    user_unique_products=("product_id", "nunique"),
    user_avg_time_since=("time_since_last", "mean")
).reset_index()

product_features = train_data.groupby("product_id").agg(
    product_total_interactions=("user_id", "count"),
    product_unique_users=("user_id", "nunique")
).reset_index()

In [7]:
# **新しい特徴量（ユーザー・商品間の関係性）**
interaction_features = train_data.groupby(["user_id", "product_id"]).agg(
    user_product_interaction_count=("event_type", "count"),
    user_product_avg_time_since=("time_since_last", "mean")
).reset_index()

In [8]:
# データ結合
train_data = train_data.merge(user_features, on="user_id", how="left")
train_data = train_data.merge(product_features, on="product_id", how="left")
train_data = train_data.merge(interaction_features, on=["user_id", "product_id"], how="left")

In [9]:
# 学習用特徴量
features = [
    "user_total_interactions", "user_unique_products", "user_avg_time_since",
    "product_total_interactions", "product_unique_users",
    "user_product_interaction_count", "user_product_avg_time_since"
]

In [10]:
# データバランス調整（リサンプリング最適化）
train_data_pos = train_data[train_data["relevance"] >= 2]  
train_data_neg = train_data[train_data["relevance"] < 2]   

train_data_neg_undersampled = resample(train_data_neg, replace=True, n_samples=700000, random_state=42)
train_data_pos_oversampled = resample(train_data_pos, replace=True, n_samples=800000, random_state=42)

train_data_balanced = pd.concat([train_data_neg_undersampled, train_data_pos_oversampled])

In [11]:
# クロスバリデーション
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ndcg_scores = []

In [12]:
for train_index, val_index in kf.split(train_data_balanced):
    train_set = train_data_balanced.iloc[train_index]
    val_set = train_data_balanced.iloc[val_index]

    X_train, y_train = train_set[features], train_set["relevance"]
    X_val, y_val = val_set[features], val_set["relevance"]

    group_train = train_set.groupby("user_id").size().to_numpy()
    group_val = val_set.groupby("user_id").size().to_numpy()

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtrain.set_group(group_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    dval.set_group(group_val)

    params = {
        "objective": "rank:ndcg",
        "eval_metric": "ndcg",
        "booster": "gbtree",
        "eta": 0.05,  # 学習率を下げて汎化性能向上
        "max_depth": 7,
        "min_child_weight": 20,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "lambda": 1.5,
        "gamma": 0.2
    }

    evals_result = {}
    model = xgb.train(params, dtrain, num_boost_round=500,
                      evals=[(dtrain, "train"), (dval, "val")],
                      evals_result=evals_result,
                      early_stopping_rounds=30,
                      verbose_eval=10)

    val_set_copy = val_set.copy()
    val_set_copy["score"] = model.predict(xgb.DMatrix(val_set_copy[features]))

    def ndcg_at_k(relevance_scores, k):
        relevance_scores = np.array(relevance_scores)
        ideal_relevance = np.sort(relevance_scores)[::-1]  # 降順ソート

        # kに満たない場合、ゼロ埋め
        if len(relevance_scores) < k:
            relevance_scores = np.pad(relevance_scores, (0, k - len(relevance_scores)), constant_values=0)
            ideal_relevance = np.pad(ideal_relevance, (0, k - len(ideal_relevance)), constant_values=0)

        dcg = np.sum(relevance_scores[:k] / np.log2(np.arange(2, k + 2)))
        ideal_dcg = np.sum(ideal_relevance[:k] / np.log2(np.arange(2, k + 2)))

        return dcg / ideal_dcg if ideal_dcg > 0 else 0

    ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()
    ndcg_scores.append(ndcg_val)

[0]	train-ndcg:0.96322	val-ndcg:0.98641
[10]	train-ndcg:0.96779	val-ndcg:0.98753
[20]	train-ndcg:0.96810	val-ndcg:0.98750
[30]	train-ndcg:0.96843	val-ndcg:0.98769
[40]	train-ndcg:0.96882	val-ndcg:0.98765
[50]	train-ndcg:0.96911	val-ndcg:0.98768
[60]	train-ndcg:0.96940	val-ndcg:0.98775
[70]	train-ndcg:0.96967	val-ndcg:0.98773
[80]	train-ndcg:0.96981	val-ndcg:0.98777
[90]	train-ndcg:0.96999	val-ndcg:0.98777
[100]	train-ndcg:0.97024	val-ndcg:0.98778
[110]	train-ndcg:0.97041	val-ndcg:0.98782
[120]	train-ndcg:0.97064	val-ndcg:0.98784
[130]	train-ndcg:0.97086	val-ndcg:0.98784
[140]	train-ndcg:0.97103	val-ndcg:0.98784
[150]	train-ndcg:0.97127	val-ndcg:0.98784
[153]	train-ndcg:0.97133	val-ndcg:0.98783


/tmp/ipykernel_25220/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.96298	val-ndcg:0.98637
[10]	train-ndcg:0.96710	val-ndcg:0.98769
[20]	train-ndcg:0.96746	val-ndcg:0.98774
[30]	train-ndcg:0.96786	val-ndcg:0.98777
[40]	train-ndcg:0.96816	val-ndcg:0.98778
[50]	train-ndcg:0.96853	val-ndcg:0.98777
[60]	train-ndcg:0.96878	val-ndcg:0.98787
[70]	train-ndcg:0.96904	val-ndcg:0.98782
[80]	train-ndcg:0.96927	val-ndcg:0.98786
[88]	train-ndcg:0.96944	val-ndcg:0.98783


/tmp/ipykernel_25220/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.96333	val-ndcg:0.98674
[10]	train-ndcg:0.96758	val-ndcg:0.98784
[20]	train-ndcg:0.96807	val-ndcg:0.98785
[30]	train-ndcg:0.96841	val-ndcg:0.98798
[40]	train-ndcg:0.96876	val-ndcg:0.98798
[50]	train-ndcg:0.96902	val-ndcg:0.98799
[60]	train-ndcg:0.96928	val-ndcg:0.98800
[70]	train-ndcg:0.96952	val-ndcg:0.98801
[80]	train-ndcg:0.96976	val-ndcg:0.98807
[90]	train-ndcg:0.97007	val-ndcg:0.98811
[100]	train-ndcg:0.97030	val-ndcg:0.98816
[110]	train-ndcg:0.97054	val-ndcg:0.98817
[120]	train-ndcg:0.97075	val-ndcg:0.98815
[130]	train-ndcg:0.97090	val-ndcg:0.98814
[136]	train-ndcg:0.97104	val-ndcg:0.98817


/tmp/ipykernel_25220/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.96265	val-ndcg:0.98653
[10]	train-ndcg:0.96709	val-ndcg:0.98759
[20]	train-ndcg:0.96737	val-ndcg:0.98766
[30]	train-ndcg:0.96783	val-ndcg:0.98765
[40]	train-ndcg:0.96816	val-ndcg:0.98772
[50]	train-ndcg:0.96854	val-ndcg:0.98770
[60]	train-ndcg:0.96872	val-ndcg:0.98773
[70]	train-ndcg:0.96896	val-ndcg:0.98770
[80]	train-ndcg:0.96917	val-ndcg:0.98775
[90]	train-ndcg:0.96944	val-ndcg:0.98778
[100]	train-ndcg:0.96960	val-ndcg:0.98785
[110]	train-ndcg:0.96983	val-ndcg:0.98782
[120]	train-ndcg:0.97000	val-ndcg:0.98783
[130]	train-ndcg:0.97024	val-ndcg:0.98785
[140]	train-ndcg:0.97035	val-ndcg:0.98785
[150]	train-ndcg:0.97059	val-ndcg:0.98789
[160]	train-ndcg:0.97076	val-ndcg:0.98791
[170]	train-ndcg:0.97094	val-ndcg:0.98792
[180]	train-ndcg:0.97113	val-ndcg:0.98789
[190]	train-ndcg:0.97130	val-ndcg:0.98791
[200]	train-ndcg:0.97153	val-ndcg:0.98796
[210]	train-ndcg:0.97166	val-ndcg:0.98798
[220]	train-ndcg:0.97182	val-ndcg:0.98804
[230]	train-ndcg:0.97194	val-ndcg:0.98802
[24

/tmp/ipykernel_25220/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


[0]	train-ndcg:0.96312	val-ndcg:0.98605
[10]	train-ndcg:0.96718	val-ndcg:0.98711
[20]	train-ndcg:0.96770	val-ndcg:0.98715
[30]	train-ndcg:0.96818	val-ndcg:0.98720
[40]	train-ndcg:0.96857	val-ndcg:0.98727
[50]	train-ndcg:0.96884	val-ndcg:0.98725
[60]	train-ndcg:0.96918	val-ndcg:0.98720
[70]	train-ndcg:0.96946	val-ndcg:0.98724
[76]	train-ndcg:0.96959	val-ndcg:0.98728


/tmp/ipykernel_25220/562300660.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_val = val_set_copy.groupby("user_id").apply(lambda x: ndcg_at_k(x["relevance"].values, k=22)).mean()


In [13]:
print(f"Average nDCG@22 from CV: {np.mean(ndcg_scores):.4f}")

Average nDCG@22 from CV: 0.4392


In [14]:
# テストデータの処理
candidate_products = train_data.groupby("product_id")["user_id"].count().reset_index()
candidate_products = candidate_products.sort_values("user_id", ascending=False).head(22)  # 上位22商品

In [15]:
# すべてのユーザーに候補商品を紐づける
test_data = test_data.assign(key=1).merge(candidate_products.assign(key=1), on="key").drop("key", axis=1)
test_data = test_data.rename(columns={'user_id_x': 'user_id'}).drop(columns=['user_id_y'])

In [16]:
# 評価データの前処理（学習データと同じ特徴量処理を適用）
test_data = test_data.merge(user_features, on="user_id", how="left")
test_data = test_data.merge(product_features, on="product_id", how="left")
test_data = test_data.merge(interaction_features, on=["user_id", "product_id"], how="left")
test_data.fillna(0, inplace=True)

In [17]:
# 予測用データ
X_test = test_data[features]
dtest = xgb.DMatrix(X_test)

In [18]:
# 予測スコアの計算
test_data["score"] = model.predict(dtest)

In [19]:
# 各ユーザーごとにランキング付け（スコアが高い順）
test_data["rank"] = test_data.groupby("user_id")["score"].rank(ascending=False, method="first")

In [20]:
# テストデータの nDCG 計算
ground_truth_test = defaultdict(dict)
for _, row in train_data.iterrows():
    ground_truth_test[row["user_id"]][row["product_id"]] = row["relevance"]

def evaluate_ndcg(data, ground_truth, k=22):
    ndcg_scores = []
    for user_id, group in data.groupby("user_id"):
        predicted_items = group.sort_values("score", ascending=False)["product_id"].tolist()
        relevance_scores = [ground_truth[user_id].get(item, 0) for item in predicted_items]
        ndcg_scores.append(ndcg_at_k(relevance_scores, k))
    return np.mean(ndcg_scores) if ndcg_scores else 0

ndcg_test = evaluate_ndcg(test_data, ground_truth_test, k=22)
print(f"Test nDCG@22: {ndcg_test:.4f}")

Test nDCG@22: 0.0248


In [21]:
# 提出用データの作成（上位 22 件のみ）
submission = test_data[test_data["rank"] <= 22][["user_id", "product_id", "rank"]]
submission['rank'] = submission['rank'].astype(int)

In [22]:
# 保存
submission.to_csv('./predict_file/predictions_D.tsv', sep="\t", index=False)